# 프로젝트 4 - Weekend 1: PDF 비정형 문서 → 텍스트 RAG

**프로젝트**: 비정형 문서(PDF/표/이미지) 기반 멀티모달 RAG 서비스 구축

**목표** — PDF에서 텍스트를 안정적으로 추출하고, 청킹·임베딩·검색·생성으로 이어지는 "텍스트 RAG 파이프라인"을 처음부터 끝까지 만든다.

**학습 목표**:
1. `PyMuPDF`와 `unstructured`로 PDF 텍스트/구조를 추출하고 전략별 차이를 이해
2. 헤더·푸터·중복 줄을 제거하는 정제 함수 작성
3. `chunk_by_title`로 의미 단위 청킹, `Document` 메타데이터 보존
4. `FAISS` 벡터스토어 구축 후 인용 출처 포함 RAG QA 함수 구현

**구성**: 문제 10개 × 약 45분 = 약 8시간

> 📦 실습 데이터는 `data/` 폴더에 이미 준비되어 있습니다. 노트북에서는 해당 PDF를 그대로 불러와 사용합니다.


In [1]:
# 환경 설정 및 라이브러리 설치
!pip install -q langchain langchain-openai langchain-community faiss-cpu \
    "unstructured[pdf]" pymupdf pdf2image tiktoken python-dotenv


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 4.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 9.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 6.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.7/107.7 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.1/543.1 kB 36.0 MB/s eta 0:00:00
  

In [5]:
import os
import time
import json
from pathlib import Path
from collections import Counter

# from dotenv import load_dotenv
# load_dotenv()

from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

# ⚠️ LangChain 패턴만 사용 — 원시 openai SDK 직접 import 금지
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, SystemMessage

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

DATA_DIR = Path("data")
PDF_REPORT = DATA_DIR / "quarterly_report.pdf"
PDF_PAPER = DATA_DIR / "research_paper.pdf"

assert PDF_REPORT.exists(), f"❌ {PDF_REPORT} 파일이 없습니다. data/ 폴더 확인."
assert PDF_PAPER.exists(), f"❌ {PDF_PAPER} 파일이 없습니다."

print(f"✅ 환경 설정 완료")
print(f"📄 보고서: {PDF_REPORT.name} ({PDF_REPORT.stat().st_size:,} bytes)")
print(f"📄 논문:   {PDF_PAPER.name} ({PDF_PAPER.stat().st_size:,} bytes)")


AssertionError: ❌ data/quarterly_report.pdf 파일이 없습니다. data/ 폴더 확인.

---
## 📦 실습 데이터

이 주말 동안 다룰 PDF는 두 가지입니다:

| 파일 | 성격 | 주요 요소 |
|------|------|----------|
| `quarterly_report.pdf` | 분기 실적 보고서 | 텍스트, 표, 차트 이미지 |
| `research_paper.pdf` | 연구 논문 | 텍스트, 그림, 수식, 표 |

데이터는 이미 `data/` 폴더에 준비되어 있어, 별도 생성 작업 없이 바로 사용할 수 있습니다.


---
## 문제 1: PyMuPDF로 PDF 메타·페이지 읽기

`PyMuPDF`(`fitz`)로 PDF를 열어 페이지 수와 페이지별 텍스트 길이를 요약하는 함수를 작성하세요.

**요구사항:**
- 함수 시그니처: `summarize_pdf(pdf_path: str) -> dict`
- 반환: `{"path", "num_pages", "total_chars", "pages": [{"page": int, "chars": int, "first_line": str}, ...]}`
- `first_line`은 각 페이지의 첫 비어있지 않은 줄(최대 60자)

**평가기준:**
- `summarize_pdf(str(PDF_REPORT))["num_pages"] >= 2`
- 각 page dict에 `page/chars/first_line` 3개 키 존재
- `first_line`이 60자 이하


In [ ]:
import fitz  # PyMuPDF

def summarize_pdf(pdf_path: str) -> dict:
    """PDF의 페이지 수와 페이지별 텍스트 통계를 반환."""
    # ---- 여기에 코드 작성 ----
    # 1) fitz.open(pdf_path)으로 문서 열기
    # 2) doc.page_count, page.get_text()
    # 3) 각 페이지 첫 비어있지 않은 줄을 first_line으로 (최대 60자)
    return {"path": pdf_path, "num_pages": 0, "total_chars": 0, "pages": []}


# 테스트
info = summarize_pdf(str(PDF_REPORT))
print(f"📄 {info['num_pages']} pages, total {info['total_chars']} chars")
for p in info["pages"][:3]:
    print(f"  p{p['page']}: {p['chars']:5d}자 │ {p['first_line']}")


---
## 문제 2: `partition_pdf("fast")` Element 추출

`unstructured`의 `partition_pdf`로 보고서 PDF를 파싱하고 Element 카테고리 통계를 만드세요.

**요구사항:**
- 함수 시그니처: `partition_fast(pdf_path: str) -> tuple[list, dict]`
- 반환: `(elements, stats)` — `stats`는 `{"total": int, "categories": Counter}`
- `strategy="fast"` 사용

**평가기준:**
- `len(elements) > 0`
- `stats["categories"]`가 `Counter` 인스턴스이고 `"Title"` 또는 `"NarrativeText"` 키 포함


In [ ]:
from unstructured.partition.pdf import partition_pdf

def partition_fast(pdf_path: str):
    """fast 전략으로 PDF를 파싱하고 카테고리 통계 반환."""
    # ---- 여기에 코드 작성 ----
    # 1) partition_pdf(filename=pdf_path, strategy="fast")
    # 2) Counter(el.category for el in elements)
    elements = []
    stats = {"total": 0, "categories": Counter()}
    return elements, stats


# 테스트
elements, stats = partition_fast(str(PDF_REPORT))
print(f"총 Element: {stats['total']}개")
for cat, cnt in stats["categories"].most_common():
    print(f"  {cat:<20}: {cnt}")


---
## 문제 3: 파싱 전략 비교 — fast vs hi_res

같은 PDF를 `fast`와 `hi_res` 두 전략으로 파싱하여 소요 시간과 Element 분포를 비교하는 함수를 작성하세요.

**요구사항:**
- 함수 시그니처: `compare_strategies(pdf_path: str, strategies=("fast", "hi_res")) -> list[dict]`
- 각 dict: `{"strategy", "elapsed_sec", "num_elements", "top_categories": list[(name, cnt)]}`
- 에러 발생 시 `{"strategy", "error"}` 형태로 기록
- 상위 카테고리는 빈도 상위 3개

**평가기준:**
- 결과 길이 == 입력 전략 수
- 적어도 하나는 `num_elements > 0` 이거나 error가 명확히 기록됨

💡 hi_res 전략은 layout detection으로 표/이미지를 잘 잡지만 GPU 없이 매우 느릴 수 있습니다.


In [ ]:
def compare_strategies(pdf_path: str, strategies=("fast", "hi_res")) -> list[dict]:
    """전략별 파싱 결과를 비교."""
    results = []
    for strategy in strategies:
        # ---- 여기에 코드 작성 ----
        # 1) time.time() 으로 시간 측정
        # 2) partition_pdf(filename=pdf_path, strategy=strategy)
        # 3) Counter로 top 3 카테고리 추출
        # 4) 에러는 {"strategy": ..., "error": str(e)}로
        results.append({"strategy": strategy, "elapsed_sec": 0.0, "num_elements": 0, "top_categories": []})
    return results


# 테스트
for r in compare_strategies(str(PDF_REPORT)):
    if "error" in r:
        print(f"❌ {r['strategy']}: {r['error']}")
    else:
        print(f"✅ {r['strategy']:<7} {r['elapsed_sec']:.2f}s, {r['num_elements']}개")
        for name, cnt in r["top_categories"]:
            print(f"     {name}: {cnt}")


---
## 문제 4: 보일러플레이트 정제 — 헤더·푸터·중복 줄 제거

추출된 텍스트에서 페이지마다 반복되는 헤더/푸터, 페이지 번호, 빈 줄을 제거하는 함수를 작성하세요.

**요구사항:**
- 함수 시그니처: `clean_text_lines(lines: list[str], min_repeat: int = 2) -> list[str]`
- 같은 줄이 `min_repeat` 이상 반복되면 제거 (헤더/푸터 후보)
- `r"^\s*[\d]{1,3}\s*$"` 같은 페이지 번호 라인 제거
- 빈 줄/공백만 있는 줄 제거
- `unicodedata.normalize("NFC", ...)` 적용

**평가기준:**
- 같은 줄이 3번 반복된 입력에 대해 모두 제거됨
- 페이지 번호 "1", "23" 같은 라인 제거


In [ ]:
import re
import unicodedata

def clean_text_lines(lines: list[str], min_repeat: int = 2) -> list[str]:
    """헤더/푸터 반복 라인, 페이지 번호, 빈 줄 제거."""
    # ---- 여기에 코드 작성 ----
    # 1) NFC 정규화
    # 2) Counter로 반복 빈도 측정
    # 3) page number 정규식 필터
    # 4) strip 후 빈 줄 필터
    return list(lines)


# 테스트
sample = [
    "Modu Tech 보고서", "1. 요약", "  ", "본문 내용 1", "Modu Tech 보고서",
    "1", "본문 내용 2", "Modu Tech 보고서", "23",
]
out = clean_text_lines(sample, min_repeat=2)
print("정제 후:", out)


---
## 문제 5: `chunk_by_title`로 의미 단위 청킹

`unstructured.chunking.title.chunk_by_title`을 사용해 Title 경계를 살린 청킹을 수행하세요.

**요구사항:**
- 함수 시그니처: `chunk_elements_by_title(elements, max_characters: int = 800) -> list`
- `chunk_by_title(elements, max_characters=max_characters, combine_text_under_n_chars=200)` 사용
- 각 chunk의 `metadata.page_number`가 살아있어야 함

**평가기준:**
- 반환 길이 >= 1
- 적어도 하나의 chunk가 `metadata.page_number is not None`


In [ ]:
from unstructured.chunking.title import chunk_by_title

def chunk_elements_by_title(elements, max_characters: int = 800):
    """Title 경계 보존 청킹."""
    # ---- 여기에 코드 작성 ----
    return []


# 테스트
elements, _ = partition_fast(str(PDF_REPORT))
chunks = chunk_elements_by_title(elements, max_characters=800)
print(f"청크 수: {len(chunks)}")
for c in chunks[:3]:
    page = getattr(c.metadata, "page_number", None)
    print(f"  [p{page}] {str(c)[:90]!r}")


---
## 문제 6: `Document` 객체로 변환 — 메타데이터 보존

청크 리스트를 LangChain `Document` 리스트로 변환하세요.

**요구사항:**
- 함수 시그니처: `chunks_to_documents(chunks, source: str) -> list[Document]`
- `metadata`에 `source`, `page_number`(없으면 `None`), `category`, `chunk_id` 포함
- `page_content`는 `str(chunk).strip()`

**평가기준:**
- 반환 길이 == 입력 청크 수
- 첫 Document에 `source`, `page_number`, `category`, `chunk_id` 4개 키 모두 존재
- `chunk_id`가 `0`부터 시작하는 일련번호


In [ ]:
def chunks_to_documents(chunks, source: str) -> list[Document]:
    """chunk → LangChain Document, metadata 풍부하게."""
    # ---- 여기에 코드 작성 ----
    docs = []
    return docs


# 테스트
docs = chunks_to_documents(chunks, source=PDF_REPORT.name)
print(f"Document 수: {len(docs)}")
print(f"첫 문서 metadata: {docs[0].metadata}")
print(f"본문(앞 80자): {docs[0].page_content[:80]!r}")


---
## 문제 7: FAISS 벡터스토어 구축

`Document` 리스트를 FAISS 인덱스로 변환하세요.

**요구사항:**
- 함수 시그니처: `build_faiss(docs: list[Document]) -> FAISS`
- `OpenAIEmbeddings(model="text-embedding-3-small")` 사용 (전역 `embeddings` 그대로)
- `FAISS.from_documents(docs, embeddings)`

**평가기준:**
- 반환값이 FAISS 인스턴스
- `"매출"` 또는 `"영업이익"` 쿼리로 검색 시 결과 length >= 1


In [ ]:
def build_faiss(docs: list[Document]) -> FAISS:
    """Document 리스트로 FAISS 벡터스토어 생성."""
    # ---- 여기에 코드 작성 ----
    return None


# 테스트
vs = build_faiss(docs)
hits = vs.similarity_search("매출", k=3)
print(f"🔍 '매출' Top-3:")
for h in hits:
    p = h.metadata.get("page_number")
    print(f"  [p{p}] {h.page_content[:80]!r}")


---
## 문제 8: 메타데이터 기반 필터 검색

특정 페이지 범위 또는 카테고리로 검색 결과를 필터링하는 함수를 작성하세요.

**요구사항:**
- 함수 시그니처: `search_filtered(vs: FAISS, query: str, k: int = 5, page_range: tuple | None = None, category: str | None = None) -> list[Document]`
- 후보를 `k*4`개 먼저 가져온 뒤 filter, 상위 `k`개 반환
- `page_range=(start, end)` (inclusive), 둘 다 만족하는 페이지만 통과
- `category`가 주어지면 metadata["category"] == category 만 통과

**평가기준:**
- `page_range=(1, 1)` 시 모든 결과의 `page_number == 1`
- filter 미사용 시 동일 query 결과와 길이 일치


In [ ]:
def search_filtered(vs: FAISS, query: str, k: int = 5, page_range=None, category=None):
    """metadata 후처리 필터 검색."""
    # ---- 여기에 코드 작성 ----
    # 1) similarity_search(query, k=k*4)
    # 2) page_range / category 필터
    # 3) 상위 k개 반환
    return []


# 테스트
out = search_filtered(vs, "고객 수", k=3, page_range=(2, 2))
for d in out:
    print(f"[p{d.metadata.get('page_number')}/{d.metadata.get('category')}] {d.page_content[:70]!r}")


---
## 문제 9: 텍스트 RAG QA — 검색 + 생성

쿼리에 대해 FAISS 검색 결과를 LLM 프롬프트에 주입하여 답변을 생성하는 함수를 작성하세요.

**요구사항:**
- 함수 시그니처: `rag_answer(vs: FAISS, query: str, k: int = 4) -> str`
- top-k 검색 결과를 컨텍스트로 사용
- 시스템 프롬프트: "주어진 컨텍스트만으로 답하라. 알 수 없으면 '문서에서 찾을 수 없음'"
- 모델: 전역 `llm` (gpt-4o-mini)

**평가기준:**
- "4분기 매출" 류 질문에 188(억) 관련 정보 포함
- 컨텍스트에 없는 질문(예: "회사 창립자는?")에는 "찾을 수 없음" 류 응답


In [ ]:
def rag_answer(vs: FAISS, query: str, k: int = 4) -> str:
    """검색 + 생성 RAG."""
    # ---- 여기에 코드 작성 ----
    # 1) vs.similarity_search(query, k=k)
    # 2) 컨텍스트 문자열 조립
    # 3) SystemMessage + HumanMessage로 llm.invoke
    return ""


# 테스트
print("Q1:", rag_answer(vs, "2025년 4분기 매출은 얼마인가요?"))
print()
print("Q2:", rag_answer(vs, "회사 창립자는 누구인가요?"))


---
## 문제 10: 출처 인용 포함 RAG — `(답변, 인용 목록)`

문제 9를 확장하여 답변과 함께 어느 페이지에서 인용했는지를 함께 반환하세요.

**요구사항:**
- 함수 시그니처: `rag_answer_with_citations(vs: FAISS, query: str, k: int = 4) -> dict`
- 반환: `{"answer": str, "citations": [{"source", "page", "snippet"}], "num_sources": int}`
- `snippet`은 본문 앞 100자
- "찾을 수 없음" 응답이면 `citations`은 빈 리스트

**평가기준:**
- "Q4 매출" 질문에 `len(citations) >= 1`
- 각 citation에 `source`/`page`/`snippet` 3 키 존재
- 모르는 질문에 빈 citations


In [ ]:
def rag_answer_with_citations(vs: FAISS, query: str, k: int = 4) -> dict:
    """답변 + 출처 인용을 함께 반환."""
    # ---- 여기에 코드 작성 ----
    # 1) similarity_search → hits
    # 2) context 조립 후 llm.invoke
    # 3) '찾을 수 없음' 응답이면 citations 비움
    return {"answer": "", "citations": [], "num_sources": 0}


# 테스트
r = rag_answer_with_citations(vs, "4분기 매출과 영업이익은?")
print(f"💬 {r['answer']}\n")
for c in r["citations"]:
    print(f"  📎 {c['source']} p{c['page']}: {c['snippet']!r}")

print("\n---\n")
r2 = rag_answer_with_citations(vs, "ModuBuds 2 가격은?")
print(f"💬 {r2['answer']}")
print(f"📎 citations: {len(r2['citations'])}건")
